# 02. 설문-참여자 매칭 파이프라인

챗봇 대화 데이터와 사전·사후 설문 데이터를 참여자 기준으로 통합

## 데이터 구성
- 챗봇 대화 및 참여자 정보 (A/B유형)
- 1차 설문, 본 평가 대상자, 종합 정보
- A/B유형 만족도 설문, 사전·사후 심리검사 (ISI·GAD-7·TEQ·PHQ-9)
- 총 9종 데이터소스

## 주요 처리
- 수집 경로가 다른 9종 데이터 전처리 및 통합
- 참여자별 챗봇 대화 이력 + 만족도 설문 + 사전·사후 심리검사 점수 통합

---
> 참여자 개인정보 보호를 위해 식별 가능한 출력값은 제거되었습니다.

# 데이터 로드 

In [3]:
import pandas as pd 
# A유형 챗봇 대화 데이터 (이름, 대화내용)
df_A = pd.read_csv("C:/Users/SIZIAI/SIAIZI/reserch/chatbot/df_A.csv") 
# A유형 챗봇 사용자 id (이름, 생년월일)
df_A_id = pd.read_csv("C:/Users/SIZIAI/SIAIZI/reserch/chatbot/df_A_id.csv") 
# A유형 챗봇 대화 데이터 (이름, 대화내용)
df_B = pd.read_csv("C:/Users/SIZIAI/SIAIZI/reserch/chatbot/df_B.csv")  
# A유형 챗봇 사용자 id (이름, 생년월일)
df_B_id = pd.read_csv("C:/Users/SIZIAI/SIAIZI/reserch/chatbot/df_B_id.csv") 

# 1차 설문 대상자 : 핸드폰, 이메일, 갤럭시워치, PHQ-9 정보 
First_Survey_Respondents  = pd.read_csv("D:/carechat/carechat_2023/data/2023_수면설문갤럭시워치_통합_1차설문대상자.csv")
# 본 평가 대상자 : 핸드폰, 이메일, 설문유형(A,B), 응답 개수, 사전 설문, 사후 설문
Main_Evaluation_Respondents  = pd.read_csv("D:/carechat/carechat_2023/data/2023_수면설문갤럭시워치_통합_본평가대상자.csv")
# 종합 : 생년월일, 핸드폰, 이메일, 성별, 유형, 사전설문, 사후설문 유무 
total_Respondents = pd.read_csv("D:/carechat/carechat_2023/data/2023_수면설문갤럭시워치_통합_종합.csv")
# A유형 설문 : 그날 대화에 대한 설문 (이메일, 생년월일, 핸드폰, 성별, 설문)
TypeA_Responses = pd.read_csv("D:/carechat/carechat_2023/data/2023_수면설문갤럭시워치_통합_A유형.csv")
# B유형 설문 : 그날 대화에 대한 설문 (이메일, 생년월일, 핸드폰, 성별, 설문)
TypeB_Responses = pd.read_csv("D:/carechat/carechat_2023/data/2023_수면설문갤럭시워치_통합_B유형.csv")
# 사전 설문 : 전화번호, 이메일, 설문
Pre_survey = pd.read_csv("D:/carechat/carechat_2023/data/2023_수면설문갤럭시워치_통합_사전설문.csv")
# 사후 설문 : 전화번호, 이메일, 헬스파일, 설문
Post_survey = pd.read_csv("D:/carechat/carechat_2023/data/2023_수면설문갤럭시워치_통합_사후설문.csv")

# 데이터 전처리 

In [4]:
First_Survey_Respondents = First_Survey_Respondents.dropna(how='all')
First_Survey_Respondents = First_Survey_Respondents.dropna(axis=1, how='all')

new_columns = [
    '타임스탬프', '핸드폰뒷자리', '이메일', 'watch_type',
    '1번', '2번', '3번', '4번', '5번', '6번',
    '7번', '8번', '9번', '확인', '총점', '본평가-대상자'
]

First_Survey_Respondents.columns = new_columns

In [ ]:
Main_Evaluation_Respondents = Main_Evaluation_Respondents.dropna(how='all')
Main_Evaluation_Respondents.columns

new_columns = [
    '타임스탬프', '핸드폰뒷자리', '이메일', 'watch_type',
    '1번', '2번', '3번', '4번', '5번', '6번',
    '7번', '8번', '9번', '확인', '총점', '본평가-대상자',
    '설문유형', '응답개수', '사전질문', '사후설문', '특이사항 '
]

Main_Evaluation_Respondents.columns = new_columns
Main_Evaluation_Respondents['핸드폰뒷자리'].fillna(0, inplace=True)
Main_Evaluation_Respondents['핸드폰뒷자리'] = Main_Evaluation_Respondents['핸드폰뒷자리'].astype(str)

In [7]:
total_Respondents = total_Respondents.dropna(how='all')
total_Respondents = total_Respondents.dropna(axis=1, how='all')

new_columns = [
    '타임스탬프', '생년월일', '핸드폰뒷자리', '이메일', '성별', '유형', '사전설문', '사후설문', '사전유무', '사후유무'
]
total_Respondents.columns = new_columns
total_Respondents['생년월일'] = total_Respondents['생년월일'].astype(int)
total_Respondents['핸드폰뒷자리'] = total_Respondents['핸드폰뒷자리'].astype(int)

In [ ]:
TypeA_Responses = TypeA_Responses.dropna(how='all')
TypeA_Responses = TypeA_Responses.dropna(axis=1, how='all')
TypeB_Responses = TypeB_Responses.dropna(how='all')
TypeB_Responses = TypeB_Responses.dropna(axis=1, how='all')

new_columns = {
    '생년월일 8자리 (Ex. 20231218)':'생년월일',
    '해드폰 뒷자리':'핸드폰뒷자리',
    '핸드폰 뒤자리':'핸드폰뒷자리',
    '성별 (신체적 , 정서적 모두 고려) 설명에 가까운 것을 선택해주세요.\n\n<보기>\n0: 신체적 남성, 정서적 남성\n1: 신체적 여성, 정서적 여성\n2: 신체적 여성, 정서적 남성\n3: 신체적 남성, 정서적 여성':'성별'
}

TypeA_Responses.rename(columns=new_columns, inplace=True)
TypeB_Responses.rename(columns=new_columns, inplace=True)

TypeA_Responses['생년월일'].fillna(0, inplace=True)
TypeA_Responses['핸드폰뒷자리'].fillna(0, inplace=True)
TypeA_Responses['성별'].fillna(0, inplace=True)
TypeB_Responses['생년월일'].fillna(0, inplace=True)
TypeB_Responses['핸드폰뒷자리'].fillna(0, inplace=True)
TypeB_Responses['성별'].fillna(0, inplace=True)

TypeA_Responses['생년월일'] = TypeA_Responses['생년월일'].astype(int)
TypeA_Responses['핸드폰뒷자리'] = TypeA_Responses['핸드폰뒷자리'].astype(int)
TypeA_Responses['성별'] = TypeA_Responses['성별'].astype(int)
TypeB_Responses['생년월일'] = TypeB_Responses['생년월일'].astype(int)
TypeB_Responses['핸드폰뒷자리'] = TypeB_Responses['핸드폰뒷자리'].astype(int)
TypeB_Responses['성별'] = TypeB_Responses['성별'].astype(int)

In [9]:
Pre_survey = Pre_survey.dropna(axis=1, how='all')
Post_survey = Post_survey.dropna(axis=1, how='all')
Pre_survey.columns[:3]

new_columns = {
    '전화번호 끝 4자리 (010-****-5678 의 경우, 5678)':'핸드폰뒷자리',
    "구글 이메일\n\n* 작업 지원 시 기입했던 메일\n* 'OOO@OOO.com'의 완전한 이메일 형식으로 작성":'이메일'
}

Pre_survey.rename(columns=new_columns, inplace=True)
Post_survey.rename(columns=new_columns, inplace=True)

# 사전, 사후 설문과 챗봇 매칭.... 

    챗봇 대화 dataframe에 채팅자를 구별할 수 있는게 username 밖에 없음. 
    1. df_A, df_B : 'username'
    2. df_A_id, df_B_id에서 'username'으로 매칭 후, 'birth' 추출
    3. total_Respondents에서 '생년월일'과 ['핸드폰 뒤자리' or '이메일] 매칭
    4. 3에서 추출한 features와  ['전화번호 끝 4자리 (010-****-5678 의 경우, 5678)',
       '구글 이메일\n\n* 작업 지원 시 기입했던 메일\n* 'OOO@OOO.com'의 완전한 이메일 형식으로 작성'] 을 매칭

    챗봇 대화 데이터 : 이름
    챗봇과 대화한 사람 id 데이터 : 이름, 생년월일
    종합 : 생년월일, 핸드폰 뒷자리, 이메일, 성별
    사전,사후 설문 : 핸드폰 뒷자리, 이메일

In [ ]:
total_Respondents

In [11]:
len(df_A_id) + len(df_B_id)

77

In [12]:
# 사전 조사 응답자 수 
len(Pre_survey.iloc[:, 2].unique())

79

In [13]:
# 사후 조사 응답자 수 
len(Post_survey.iloc[:, 2].unique())

58

In [14]:
df_A.columns

Index(['Unnamed: 0', 'id_msg', 'username', 'person', 'chatbot'], dtype='object')

In [15]:
df_B.columns

Index(['Unnamed: 0', 'id_msg', 'username', 'person', 'chatbot', 'date'], dtype='object')

In [16]:
df_B.iloc[:, -1]

0       2023-12-20 16:31:47
1       2023-12-20 16:32:29
2       2023-12-20 16:33:12
3       2023-12-20 16:33:40
4       2023-12-20 16:33:54
               ...         
1347    2023-12-30 09:25:59
1348    2023-12-30 09:26:55
1349    2023-12-30 09:28:32
1350    2023-12-30 09:29:56
1351    2023-12-30 09:30:37
Name: date, Length: 1352, dtype: object

In [17]:
Pre_survey.iloc[:,0]

0                     NaN
1     12-19-2023 19:07:09
2     12-19-2023 19:08:02
3     12-19-2023 19:10:20
4     12-19-2023 19:11:22
             ...         
77      1-6-2024 13:54:49
78      1-6-2024 14:15:09
79      1-6-2024 14:38:48
80      1-6-2024 14:39:59
81      1-8-2024 17:00:53
Name: 타임스탬프, Length: 82, dtype: object

In [18]:
Post_survey.iloc[:,0]

0     12-19-2023 21:07:57
1     12-21-2023 21:41:32
2     12-25-2023 22:55:00
3     12-27-2023 20:57:26
4     12-28-2023 14:58:06
             ...         
57     1-11-2024 17:32:53
58     1-11-2024 22:54:29
59      1-12-2024 8:43:58
60     1-12-2024 21:07:26
61     1-14-2024 21:42:50
Name: 타임스탬프, Length: 62, dtype: object

In [19]:
df_A_id.columns

Index(['Unnamed: 0', 'id_msg', 'username', 'birth', 'id_thread'], dtype='object')

In [20]:
df_B_id.columns

Index(['Unnamed: 0', 'id_msg', 'username', 'birth', 'id_thread'], dtype='object')

In [21]:
total_Respondents.columns

Index(['타임스탬프', '생년월일', '핸드폰뒷자리', '이메일', '성별', '유형', '사전설문', '사후설문', '사전유무',
       '사후유무'],
      dtype='object')

In [22]:
Pre_survey.columns


Index(['타임스탬프', '핸드폰뒷자리', '이메일', '동의합니다.',
       '제공항목은 설문지의 일별 작업자 매칭을 위해서 제공하며, 다른 목적으로 사용하지 않습니다.',
       '1. 불면증에 관한 문제들의 현재 (즉, 최근 2주간) 심한 정도를 표시해 주세요.\n\n1-a. 잠들기 어렵다.\n\n<보기>\n0점 : 없다\n1점 : 약간 정도\n2점 : 중간 정도\n3점 : 심하다\n4점 : 매우 심하다',
       '1. 불면증에 관한 문제들의 현재 (즉, 최근 2주간) 심한 정도를 표시해 주세요. \n\n1-b. 잠을 유지하기 어렵다.\n\n<보기>\n0점 : 없다\n1점 : 약간 정도\n2점 : 중간 정도\n3점 : 심하다\n4점 : 매우 심하다',
       '1. 불면증에 관한 문제들의 현재 (즉, 최근 2주간) 심한 정도를 표시해 주세요.\n\n1-c. 쉽게 깬다.\n\n<보기>\n0점 : 없다\n1점 : 약간 정도\n2점 : 중간 정도\n3점 : 심하다\n4점 : 매우 심하다',
       '2. 현재 수면 양상에 관하여 얼마나 만족하고 있습니까?\n\n<보기>\n0점 : 매우 만족\n1점 : 약간 만족\n2점 : 그저 그렇다\n3점 : 약간 불만족\n4점 : 매우 불만족',
       '3. 불면증이 낮 활동을 어느 정도 방해한다고 생각합니까? (예: 낮에 피곤함, 직장이나 가사에 일하는 능력, 집중력, 기억력, 기분, 등)\n\n<보기>\n0점 : 전혀 방해되지 않는다\n1점 : 약간\n2점 : 다소\n3점 : 상당히\n4점 : 매우 많이',
       '4. 불면증으로 인한 삶의 질 손상 정도가 다른 사람들에게 어떻게 보인다고 생각합니까?\n\n<보기>\n0점 : 전혀 그렇게 보이지 않는다\n1점 : 약간\n2점 : 다소\n3점 : 상당히\n4점 : 매우 심하게 보인다',
       '5. 현재의 수면 장애에 관하여 얼마나 걱정하고 있습니까?\n\n<보기>\n0점 : 전혀 걱정하지 않는다\n1점 : 약간\n2

In [23]:
Pre_survey.iloc[:, 0:3].columns

Index(['타임스탬프', '핸드폰뒷자리', '이메일'], dtype='object')

In [24]:
Post_survey.iloc[:,0]

0     12-19-2023 21:07:57
1     12-21-2023 21:41:32
2     12-25-2023 22:55:00
3     12-27-2023 20:57:26
4     12-28-2023 14:58:06
             ...         
57     1-11-2024 17:32:53
58     1-11-2024 22:54:29
59      1-12-2024 8:43:58
60     1-12-2024 21:07:26
61     1-14-2024 21:42:50
Name: 타임스탬프, Length: 62, dtype: object

In [25]:
Post_survey.iloc[:, 0:3].columns

Index(['타임스탬프', '핸드폰뒷자리', '이메일'], dtype='object')

In [ ]:
df_A['username'].unique()

## total - 이름, 생일, 이메일, 핸드폰, 성별 

In [ ]:
df_TypeA_Responses = df_A_id[['username', 'birth']]
df_TypeA_Responses.columns = ['이름', '생년월일']
df_TypeA_Responses = pd.merge(df_TypeA_Responses, TypeA_Responses[['생년월일', '이메일', '핸드폰뒷자리', '성별']], on='생년월일', how='left')
df_TypeA_Responses.drop_duplicates(inplace=True)
df_TypeA_Responses

In [ ]:
df_TypeA_Responses.to_csv('TypeA_Responses.csv')

In [ ]:
df_TypeB_Responses = df_B_id[['username', 'birth']]
df_TypeB_Responses.columns = ['이름', '생년월일']
df_TypeB_Responses = pd.merge(df_TypeB_Responses, TypeB_Responses[['생년월일', '이메일', '핸드폰뒷자리', '성별']], on='생년월일', how='left')
df_TypeB_Responses.drop_duplicates(inplace=True)
df_TypeB_Responses

In [ ]:
df_TypeB_Responses.to_csv('TypeB_Responses.csv')

In [29]:
len(TypeA_Responses)

240

In [30]:
A = total_Respondents.loc[total_Respondents['유형']=='A유형']
len(A)
# A_count = A['생년월일'].value_counts()

236

In [31]:
len(TypeB_Responses)

254

In [32]:
B = total_Respondents.loc[total_Respondents['유형']=='B유형']
len(B)

252

In [ ]:
total_Respondents

## 데이터 처리

In [34]:
df_TypeA_Responses.columns

Index(['이름', '생년월일', '이메일', '핸드폰뒷자리', '성별'], dtype='object')

In [ ]:
df_TypeB_Responses

In [36]:
total_Respondents.columns

Index(['타임스탬프', '생년월일', '핸드폰뒷자리', '이메일', '성별', '유형', '사전설문', '사후설문', '사전유무',
       '사후유무'],
      dtype='object')

# 대화내용과 매칭

## df_A

In [ ]:
total_A = df_TypeA_Responses[['이름', '생년월일']].copy()
total_A = total_A.merge(total_Respondents[['생년월일', '핸드폰뒷자리', '이메일', '성별', '유형', '사전설문', '사후설문', '사전유무', '사후유무']],
                        on='생년월일', how='left')
total_A = total_A.drop_duplicates()
total_A

In [37]:
names_with_nan_email = total_A[total_A['이메일'].isna()]['이름'].tolist()

In [ ]:
# total_A에서 username이 names_with_nan_email에 포함되지 않은 행만 남김
total_A_filtered = total_A[~total_A['이름'].isin(names_with_nan_email)]
total_A_filtered

In [ ]:
# df_A에서 username이 names_with_nan_email에 포함되지 않은 행만 남김
df_A_filtered = df_A[~df_A['username'].isin(names_with_nan_email)]
df_A_filtered

In [ ]:
df_A[df_A['username']=='유지순']

In [41]:
df_A_filtered[df_A_filtered['username']=='유지순']

,Unnamed: 0,id_msg,username,person,chatbot


## df_B

In [ ]:
total_B = df_TypeB_Responses[['이름', '생년월일']].copy()
total_B = total_B.merge(total_Respondents[['생년월일', '핸드폰뒷자리', '이메일', '성별', '유형', '사전설문', '사후설문', '사전유무', '사후유무']],
                        on='생년월일', how='left')
total_B = total_B.drop_duplicates()
total_B

In [ ]:
df_B[df_B['username']=='소정아']

In [44]:
total_Respondents[total_Respondents['생년월일']==19780821]

,타임스탬프,생년월일,핸드폰뒷자리,이메일,성별,유형,사전설문,사후설문,사전유무,사후유무


In [45]:
TypeB_Responses[TypeB_Responses['생년월일']==19780821].iloc[:,:5]

,타임스탬프,이메일,생년월일,핸드폰뒷자리,성별


In [46]:
df_B.columns

Index(['Unnamed: 0', 'id_msg', 'username', 'person', 'chatbot', 'date'], dtype='object')

In [47]:
total_B.columns

Index(['이름', '생년월일', '핸드폰뒷자리', '이메일', '성별', '유형', '사전설문', '사후설문', '사전유무',
       '사후유무'],
      dtype='object')

In [48]:
names_with_nan_email = total_B[total_B['이메일'].isna()]['이름'].tolist()

In [ ]:
# total_A에서 username이 names_with_nan_email에 포함되지 않은 행만 남김
total_B_filtered = total_B[~total_B['이름'].isin(names_with_nan_email)]
total_B_filtered

In [ ]:
# df_B에서 username이 names_with_nan_email에 포함되지 않은 행만 남김
df_B_filtered = df_B[~df_B['username'].isin(names_with_nan_email)]
df_B_filtered

In [51]:
df_B_filtered[df_B_filtered['username']=='소정아']

,Unnamed: 0,id_msg,username,person,chatbot,date
